# Power BI 設計規格 — 04_Inventory_Performance_Analysis

> **專案背景**：資料來源為 PwC × Kaggle Inventory Analysis，透過 PostgreSQL ELT 建立 Raw → Staging → Marts 三層架構，最終產出 Star Schema（`dim_product`、`dim_store`、`dim_vendor`、`dim_date`、`fact_sales`、`fact_inventory_snapshot`），再以 Power BI Import Mode 消費。目標 KPI 為 Inventory Turnover、DSI、Stockout Rate、Dead Stock、ABC Classification 與 Reorder Point。

***

## 一、資料約束與設計前提

在動手做任何 Visual 之前，先釐清資料的真實限制，避免設計出無法支撐的 KPI：

| 資料表 | 時間點/範圍 | 說明 |
|---|---|---|
| `raw_beg_inventory` | 期初一個快照（2016-01-01）| 約 20 萬筆 |
| `raw_end_inventory` | 期末一個快照（2016-12-31）| 約 20 萬筆 |
| `raw_sales` | 2016-01-01 ~ 2016-12-31，完整 366 天 | 1,200 萬+ 筆 |
| `fact_inventory_snapshot` | BEGINNING + ENDING 兩列 per SKU/Store | 不能計算中間月份庫存趨勢 |

**關鍵限制**：`fact_inventory_snapshot` 只有期初與期末兩個時間點，因此「月度庫存趨勢線」在本資料集中**無法精準計算**，設計時必須誠實面對這個限制。但 `fact_sales` 有完整 366 天，可計算月度銷售趨勢、月均銷售量、Reorder Point 等。

***

## 二、資料模型（Power BI Relationship 設定）

Power BI Import Mode 下，嚴格遵守 Star Schema 單向篩選，避免雙向 Cross-filter 造成 Ambiguity 問題：

| From Table (Many) | From Column | To Table (One) | To Column | Cardinality | Direction |
|---|---|---|---|---|---|
| `fact_sales` | `product_sk` | `dim_product` | `product_sk` | Many-to-One | Single → |
| `fact_sales` | `store_sk` | `dim_store` | `store_sk` | Many-to-One | Single → |
| `fact_sales` | `vendor_sk` | `dim_vendor` | `vendor_sk` | Many-to-One | Single → |
| `fact_sales` | `sales_date` | `dim_date` | `date_key` | Many-to-One | Single → |
| `fact_inventory_snapshot` | `product_sk` | `dim_product` | `product_sk` | Many-to-One | Single → |
| `fact_inventory_snapshot` | `store_sk` | `dim_store` | `store_sk` | Many-to-One | Single → |
| `fact_inventory_snapshot` | `snapshot_date` | `dim_date` | `date_key` | Many-to-One | Single → |

**重要設定**：`dim_date` 建立後，在 Power BI Model View 右鍵 → **Mark as Date Table**（Date column = `date_key`），否則 `DATESYTD`、`DATEADD` 等 Time Intelligence 函數不會生效。

***

## 三、完整 DAX Measure 規格

所有 Measures 建議集中在一張獨立 `_Measures` 表（新增空白表，不 import 任何資料），方便管理與 GitHub 文件化。

### Group 01：基礎數值

```dax
-- 總銷售額
Total Revenue =
SUMX('fact_sales', 'fact_sales'[sales_dollars])

-- 總銷售量
Total Sales Qty =
SUM('fact_sales'[sales_quantity])

-- 估算 COGS（銷售量 × 成本價）
Total COGS =
SUMX('fact_sales', 'fact_sales'[estimated_cogs])

-- 期初庫存總值
Beginning Inventory Value =
CALCULATE(
    SUM('fact_inventory_snapshot'[total_inventory_value]),
    'fact_inventory_snapshot'[snapshot_type] = "BEGINNING"
)

-- 期末庫存總值
Ending Inventory Value =
CALCULATE(
    SUM('fact_inventory_snapshot'[total_inventory_value]),
    'fact_inventory_snapshot'[snapshot_type] = "ENDING"
)

-- 期末庫存量（在手）
Current Stock Qty =
CALCULATE(
    SUM('fact_inventory_snapshot'[quantity_on_hand]),
    'fact_inventory_snapshot'[snapshot_type] = "ENDING"
)

-- 平均庫存值（期初+期末 / 2）
Average Inventory Value =
DIVIDE(
    [Beginning Inventory Value] + [Ending Inventory Value],
    2
)
```

### Group 02：庫存周轉 KPI

$$
\text{Inventory Turnover} 
= \frac{\text{COGS (期間銷售成本)}}{\text{Average Inventory}} 
$$



```dax
-- 庫存周轉率（年度）
Inventory Turnover =
DIVIDE(
    [Total COGS],
    [Average Inventory Value],
    BLANK()
)
```
$$
    \text{DSI} = \frac{365}{\text{Inventory Turnover}} 
$$

### Step 1：基礎 Measure（建議建在獨立 `_Measures` 表中）

```
-- DSI（庫存銷售天數）
Days Sales of Inventory (DSI) =
VAR _turnover = [Inventory Turnover]
RETURN
    IF(
        _turnover = 0 || ISBLANK(_turnover),
        BLANK(),
        DIVIDE(365, _turnover)
    )

-- YTD 版本（搭配 dim_date Time Intelligence）
Inventory Turnover YTD =
CALCULATE(
    [Inventory Turnover],
    DATESYTD('dim_date'[date_key])
)
```

> ⚠️ **資料限制提示**：由於庫存快照只有期初/期末兩個時間點，`Inventory Turnover` 與 `DSI` 在月度視圖下會顯示相同值（分母 Average Inventory 不隨月份變動）。建議在 Page 1 的圖表 subtitle 加入說明，或限制此 KPI 只在年度層級顯示。

### Group 03：缺貨率

```dax
-- 缺貨快照記錄數
Out of Stock Records =
CALCULATE(
    [Total Snapshot Records],
    'fact_inventory_snapshot'[quantity_on_hand] <= 0
)

-- 總快照記錄數
Total Snapshot Records =
COUNTROWS('fact_inventory_snapshot')

-- 缺貨率（以快照比例定義）
Stockout Rate = 
VAR _oos =
    CALCULATE(
        COUNTROWS( 'fact_inventory_snapshot'),
        'fact_inventory_snapshot'[quantity_on_hand] <= 0,
        'fact_inventory_snapshot'[snapshot_type] = "ENDING"
    )
VAR _total =
    CALCULATE(
        COUNTROWS(' fact_inventory_snapshot'),
        'fact_inventory_snapshot'[snapshot_type] = "ENDING"
    )
RETURN DIVIDE(_oos, _total, BLANK())
```

### Group 04：呆滯庫存

```dax
-- 過去 90 天銷售量（以 dim_date 為基準）
Sales Qty Last 90 Days =
CALCULATE(
    SUM('fact_sales'[sales_quantity]),
    DATESINPERIOD(
        'dim_date'[date_key],
        MAX('dim_date'[date_key]),
        -90,
        DAY
    )
)

-- 呆滯庫存金額（90 天無銷售 + 期末有庫存）
Dead Stock Value (90 Days) =
CALCULATE(
    [Ending Inventory Value],
    FILTER(
        'dim_product',
        [Sales Qty Last 90 Days] = 0 || ISBLANK([Sales Qty Last 90 Days])
    ),
    'fact_inventory_snapshot'[quantity_on_hand] > 0
)

-- 呆滯庫存 SKU 數
Dead Stock SKU Count =
CALCULATE(
    DISTINCTCOUNT('dim_product'[product_sk]),
    FILTER(
        'dim_product',
        ( [Sales Qty Last 90 Days] = 0 || ISBLANK([Sales Qty Last 90 Days]) )
        &&
        CALCULATE(
            SUM('fact_inventory_snapshot'[quantity_on_hand]),
            'fact_inventory_snapshot'[snapshot_type] = "ENDING"
        ) > 0
    )
)

-- 呆滯庫存占比（占期末總庫存金額）
Dead Stock % of Total Inventory =
DIVIDE([Dead Stock Value (90 Days)], [Ending Inventory Value], BLANK())

-- 呆滯旗標（表格用）
Dead Stock Flag =
IF([Dead Stock Value (90 Days)] > 0, "Dead Stock", "Active")
```

> ⚠️ **資料限制說明**：本資料集期末快照日期為 2016-12-31，因此「過去 90 天」的計算實際上是 2016-10-03 ~ 2016-12-31。任何篩選上下文改變 `MAX('dim_date'[date_key])` 時，90 天視窗會跟著移動，邏輯正確。

### Group 05：再訂購點

```dax
-- 日均銷售量
Avg Daily Sales Qty =
VAR _totalQty   = SUM('fact_sales'[sales_quantity])
VAR _activeDays =
    CALCULATE(
        DISTINCTCOUNT('dim_date'[date_key]),
        CROSSFILTER('fact_sales'[sales_date], 'dim_date'[date_key], BOTH)
    )
RETURN
    DIVIDE(_totalQty, _activeDays, BLANK())

-- 日銷售量標準差（統計安全庫存用）
Stddev Daily Sales Qty =
VAR _salesByDay =
    ADDCOLUMNS(
        VALUES('dim_date'[date_key]),
        "@DailySales",
        CALCULATE(SUM('fact_sales'[sales_quantity]))
    )
VAR _avgDaily = AVERAGEX(_salesByDay, [@DailySales])
VAR _variance =
    AVERAGEX(
        _salesByDay,
        ([@DailySales] - _avgDaily) ^ 2
    )
RETURN SQRT(_variance)

-- 安全庫存量（Z=1.65, 95% 服務水準, Lead Time=7 天）
Safety Stock Qty =
VAR _leadTimeDays = 7
VAR _zScore       = 1.65
RETURN
    _zScore * [Stddev Daily Sales Qty] * SQRT(_leadTimeDays)

-- 再訂購點數量
Reorder Point Qty =
VAR _leadTimeDays       = 7
VAR _demandDuringLead   = [Avg Daily Sales Qty] * _leadTimeDays
RETURN
    ROUND(_demandDuringLead + [Safety Stock Qty], 0)

-- 補貨警示（四色旗標）
Reorder Alert =
VAR _currentStock = [Current Stock Qty]
VAR _rop          = [Reorder Point Qty]
RETURN
    SWITCH(
        TRUE(),
        ISBLANK(_rop) || ISBLANK(_currentStock), BLANK(),
        _currentStock <= 0,          "🔴 Out of Stock",
        _currentStock <= _rop,       "🟡 Reorder Now",
        _currentStock <= _rop * 1.2, "🟠 Low Stock",
        "🟢 OK"
    )

-- 需補貨 SKU 總數（Page 3 Card 用）
Reorder SKU Count =
CALCULATE(
    DISTINCTCOUNT('dim_product'[product_sk]),
    FILTER(
        'dim_product',
        [Reorder Alert] = "🟡 Reorder Now"
    )
)
```

### Group 06：ABC 輔助欄位（Calculated Columns on `dim_product`）

以下為 Calculated Column（不是 Measure），直接加在 `dim_product` 表：

```dax
-- 顯示用欄位（消除 Blank 軸標籤）
ABC Class Display =
COALESCE('dim_product'[abc_class], "Unclassified")

-- 排序欄位（設定 "Sort by column" = 此欄）
ABC Class Sort =
SWITCH(
    TRUE(),
    'dim_product'[abc_class] = "A", 1,
    'dim_product'[abc_class] = "B", 2,
    'dim_product'[abc_class] = "C", 3,
    4  -- Unclassified 排最後
)

-- 業務語言版本（Page 3 矩陣用）
ABC Priority Group =
SWITCH(
    TRUE(),
    'dim_product'[abc_class] = "A", "A - Critical",
    'dim_product'[abc_class] = "B", "B - Important",
    'dim_product'[abc_class] = "C", "C - Long Tail",
    "Review - Unclassified"
)
```

### Group 07：未分類庫存監控

```dax
Unclassified SKU Count =
CALCULATE(
    DISTINCTCOUNT('dim_product'[product_sk]),
    FILTER('dim_product', ISBLANK('dim_product'[abc_class]))
)

Unclassified Inventory Value =
CALCULATE(
    [Ending Inventory Value],
    FILTER('dim_product', ISBLANK('dim_product'[abc_class]))
)

Unclassified Dead Stock Value =
CALCULATE(
    [Dead Stock Value (90 Days)],
    FILTER('dim_product', ISBLANK('dim_product'[abc_class]))
)
```

### Group 08：動態圖表標題

```dax
Title - Dead Stock by ABC =
VAR _store = SELECTEDVALUE('dim_store'[store_number], "All Stores")
VAR _year  = SELECTEDVALUE('dim_date'[year], "All Years")
RETURN
    "Dead Stock Value (90 Days) by ABC Class | Store: " & _store & " | Year: " & _year

Title - Reorder Matrix =
VAR _abc   = SELECTEDVALUE('dim_product'[abc_class], "All Classes")
VAR _store = SELECTEDVALUE('dim_store'[store_number], "All Stores")
RETURN
    "Replenishment Status | ABC: " & _abc & " | Store: " & _store
```

***

## 四、三頁報表設計規格

### 全域設計原則

- **Slicer 組合（所有頁面一致）**：`Year`、`Month`、`Store`、`Vendor`、`ABC Class`；Page 2 & 3 額外加 `Product Search`（Search Slicer 類型）
- **色彩規則**：正常用中性灰藍，警示僅用少量強調色：🔴 Out of Stock / 🟡 Reorder Now / 🟠 Low Stock / 🟢 OK
- **導航按鈕**：每頁右上角固定放 Page 1 / Page 2 / Page 3 按鈕 + `Last Refresh Date` Card
- **Drill-through 設定**：Page 1 點擊 Store → Drill-through 到 Page 2；點擊 Product → Drill-through 到 Page 3

***

### Page 1 — Executive Overview

**核心問題**：*整體庫存績效健不健康？哪個方向最值得優先追蹤？*

#### Wireframe

```
┌──────────────────────────────────────────────────────────────────────┐
│ [Nav: 1 Overview | 2 Inventory Risk | 3 Replenishment]   [Refresh]  │
│ Title: Inventory Performance Overview · FY 2016                      │
│ Slicers: Year | Month | Store | Vendor | ABC Class                   │
├──────────────────────────────────────────────────────────────────────┤
│  KPI Card 1       │ KPI Card 2       │ KPI Card 3    │ KPI Card 4   │
│  Inventory        │ DSI              │ Stockout Rate │ Dead Stock % │
│  Turnover         │ (Days)           │ (%)           │ of Inventory │
├──────────────────────────────────────────────────────────────────────┤
│  Line: Monthly Total Revenue & COGS Trend  (fact_sales 有完整366天) │
├────────────────────────────────────┬─────────────────────────────────┤
│  Column: Stockout Rate by Store    │ Bar: Dead Stock Value by ABC    │
│  (排序高→低, 抓風險門店)            │ (突出 C 類積壓)                 │
├────────────────────────────────────┴─────────────────────────────────┤
│  Matrix: Store × ABC Class / Revenue / End. Inv / Turnover / DSI    │
└──────────────────────────────────────────────────────────────────────┘
```

#### Visual 規格

| Visual | Type | Axis / Legend | Values | 備註 |
|---|---|---|---|---|
| KPI Card 1 | Card (New) | — | `Inventory Turnover` | Subtitle: "FY 2016 Estimate" |
| KPI Card 2 | Card | — | `Days Sales of Inventory (DSI)` | Format: 0 days |
| KPI Card 3 | Card | — | `Stockout Rate` | Format: % |
| KPI Card 4 | Card | — | `Dead Stock % of Total Inventory` | 目標線 5%，紅色警示 |
| Monthly Trend | Line | `dim_date[month_name_short]` | `Total Revenue`, `Total COGS` | 雙線，Revenue 藍 / COGS 橙 |
| Stockout by Store | Clustered Column | `dim_store[store_number]` | `Stockout Rate` | 依高→低排序 |
| Dead Stock by ABC | Bar | `dim_product[ABC Class Display]` | `Dead Stock Value (90 Days)` | 使用動態 Title Measure |
| Summary Matrix | Matrix | Rows: `Store`, `ABC Class Display` | Revenue / End. Inv Value / Turnover / DSI / Stockout Rate | Conditional Format: Turnover 低→紅 |

> **設計決策說明**：月度趨勢選擇 Revenue + COGS（來自 `fact_sales`），而非 Inventory Turnover，因為後者的分母庫存值只有年初/年末兩點，月度顯示無意義。誠實呈現資料限制是作品集的加分項。

***

### Page 2 — Inventory Risk Analysis

**核心問題**：*哪些商品或門店同時面臨缺貨與呆滯的雙重風險？*

#### Wireframe

```
┌──────────────────────────────────────────────────────────────────────┐
│ [Nav] [Refresh]                                                      │
│ Title: Inventory Risk Analysis                                       │
│ Slicers: Year | Month | Store | Vendor | ABC Class | Product Search │
├──────────────────────────────────────────────────────────────────────┤
│  Card: Dead Stock Value  │ Card: Dead Stock SKU # │ Card: Out of Stock Records    │
├───────────────────────────────────────┬──────────────────────────────┤
│  Scatter: Stockout Rate vs            │ Treemap/Bar:                 │
│  Dead Stock % (by Store or Category) │ Dead Stock by Vendor         │
├───────────────────────────────────────┴──────────────────────────────┤
│  Detail Table: Product / Store / ABC / QOH / Sales90D / Value / Flag│
└──────────────────────────────────────────────────────────────────────┘
```

#### Visual 規格

| Visual | Type | 設定 |
|---|---|---|
| Dead Stock Value Card | Card | `Dead Stock Value (90 Days)` |
| Dead Stock SKU Count Card | Card | `Dead Stock SKU Count` |
| OOS Records Card | Card | `Out of Stock Records` |
| Risk Scatter | Scatter | X: `Stockout Rate` / Y: `Dead Stock % of Total Inventory` / Size: `Ending Inventory Value` / Details: `dim_store[store_number]` |
| Dead Stock by Vendor | Bar | Axis: `dim_vendor[vendor_name]` / Value: `Dead Stock Value (90 Days)` / Top N: 15 |
| Risk Detail Table | Table | `dim_product[brand]` / `dim_store[store_number]` / `ABC Class Display` / `Current Stock Qty` / `Sales Qty Last 90 Days` / `Dead Stock Value (90 Days)` / `Dead Stock Flag` |

**Conditional Formatting for Detail Table**：
- `Dead Stock Flag` = "Dead Stock" → 背景色紅（#FFE0E0）
- `Current Stock Qty` = 0 → 字體色紅

***

### Page 3 — Replenishment & ABC Prioritization

**核心問題**：*哪些 SKU 現在該補貨，按 ABC 優先順序如何排？*

#### Wireframe

```
┌──────────────────────────────────────────────────────────────────────┐
│ [Nav] [Refresh]                                                      │
│ Title: [Title - Reorder Matrix 動態標題]                             │
│ Slicers: Year | Month | Store | Vendor | ABC Class | Product Search │
├──────────────────────────────────────────────────────────────────────┤
│  Card: Reorder SKU Count │ Card: Avg Daily Sales │ Card: End. Inv $ │
├───────────────────────────────────────┬──────────────────────────────┤
│  Bar: Reorder Alert Count by ABC      │ Bar: Top 15 Reorder Products │
│  (🟡+🔴 的 SKU 數，按 ABC 分組)      │ (Filter: Reorder Now only)   │
├───────────────────────────────────────┴──────────────────────────────┤
│  Matrix: Product / Store / ABC / QOH / AvgDaily / Safety / ROP /    │
│          Reorder Alert / Vendor                                       │
└──────────────────────────────────────────────────────────────────────┘
```

#### Visual 規格

| Visual | Type | 設定 |
|---|---|---|
| Reorder SKU Count | Card | `Reorder SKU Count` |
| Avg Daily Sales | Card | `Avg Daily Sales Qty` / Format: 0.0 |
| Ending Inventory Value | Card | `Ending Inventory Value` |
| Alert by ABC | Clustered Column | Axis: `ABC Class Display` / Value: `Reorder SKU Count` / Legend: `Reorder Alert` / Sort: `ABC Class Sort` |
| Top Reorder Products | Bar | Axis: `dim_product[brand]` / Value: `Reorder Point Qty` / Visual Level Filter: `Reorder Alert = "🟡 Reorder Now"` / Top N: 15 |
| Main Replenishment Matrix | Matrix | Rows: `ABC Priority Group`, `brand`, `store_number` / Values: `Current Stock Qty`, `Avg Daily Sales Qty`, `Safety Stock Qty`, `Reorder Point Qty`, `Reorder Alert` / Conditional Format: Reorder Alert 依值設色 |

**矩陣 Conditional Formatting 規則**（`Reorder Alert` 欄）：
```
🔴 Out of Stock  → 背景 #FF0000, 字 白色
🟡 Reorder Now   → 背景 #FFD700, 字 黑色
🟠 Low Stock     → 背景 #FFA500, 字 黑色
🟢 OK            → 背景 #E8F5E9, 字 黑色
```

***

## 五、Tooltip 設計

高價值 Visual 建議加上 Report Page Tooltip，讓使用者 hover 時看到 SKU 層級細節：

### Tooltip Page — SKU Detail（5×3 cm 小頁面）

Visuals：
- `ABC Priority Group`（Text）
- `Avg Daily Sales Qty`（Card）
- `Safety Stock Qty`（Card）
- `Reorder Point Qty`（Card）
- `Current Stock Qty`（Card）
- `Reorder Alert`（Card）

在 Page 3 矩陣的 `brand` 欄位設定此 Tooltip Page。

***

## 六、建議補充的 Calculated Column（`dim_date`）

`dim_date` 需確保以下欄位存在，供 Slicer 與 Axis 使用：

```dax
-- 月份短名（Slicer / Axis）
month_name_short = FORMAT('dim_date'[date_key], "MMM")

-- 月份排序
month_sort = MONTH('dim_date'[date_key])

-- 年月組合（Trend Axis）
year_month = FORMAT('dim_date'[date_key], "YYYY-MM")

-- 季度
quarter = "Q" & QUARTER('dim_date'[date_key])
```

設定 `month_name_short` 的 "Sort by Column" = `month_sort`，否則月份軸會按字母排成 Apr / Aug / Dec...。

***

## 七、Measure Group 建議清單（`_Measures` 表）

在 Power BI 的 Model View 用 **Display Folder** 組織所有 Measures，方便日後維護：

| Display Folder | Measures |
|---|---|
| `01_Base` | Total Revenue, Total Sales Qty, Total COGS, Beginning/Ending Inventory Value, Current Stock Qty, Average Inventory Value |
| `02_Turnover` | Inventory Turnover, Days Sales of Inventory (DSI), Inventory Turnover YTD |
| `03_Stockout` | Out of Stock Records, Total Snapshot Records, Stockout Rate |
| `04_Dead Stock` | Sales Qty Last 90 Days, Dead Stock Value (90 Days), Dead Stock SKU Count, Dead Stock % of Total Inventory, Dead Stock Flag |
| `05_Reorder` | Avg Daily Sales Qty, Stddev Daily Sales Qty, Safety Stock Qty, Reorder Point Qty, Reorder Alert, Reorder SKU Count |
| `06_Unclassified` | Unclassified SKU Count, Unclassified Inventory Value, Unclassified Dead Stock Value |
| `07_Titles` | Title - Dead Stock by ABC, Title - Reorder Matrix |

***

## 八、作品集說明文字建議（README / 報表 Info Button）

為每個 KPI 加上「i」按鈕 Tooltip，說明計算邏輯與資料假設，能展示對方法論的掌握。建議文字如下：

**Inventory Turnover**：COGS / 平均庫存值（期初+期末除以2）。本資料集庫存快照僅有期初期末兩點，故此指標為年度估算。

**DSI**：365 / Inventory Turnover。同上，為年度層級最準確。

**Stockout Rate**：庫存量 ≤ 0 的快照記錄數 / 總快照記錄數。以快照比例定義，非以 SKU 天數。

**Dead Stock**：過去 90 天零銷售 且 期末庫存 > 0 的 SKU 庫存金額。90 天窗口自最後一筆銷售日動態往前推算。

**Reorder Point**：(日均銷售量 × 前置天數) + 安全庫存。安全庫存採 Z=1.65（95% 服務水準）× 銷售標準差 × √前置天數（7 天）。前置天數預設為 7 天，無原始資料依據，屬業界假設。

**ABC Classification**：依全年銷售額計算累積佔比，前 80% 為 A、80–95% 為 B、後 5% 為 C。在 PostgreSQL 預計算後寫入 `dim_product.abc_class`，避免 Power BI 對 1,200 萬筆 fact 表進行 Running Total 導致效能問題。


## 問題解答：空白 vs 0.00 的差別
在 Page 2 的 Risk Detail Table 中，`Current Stock Qty` 欄位出現「空白（Blank）」和「0.00」是兩種完全不同的資料狀態，Power BI / DAX 明確區分這兩者 。
### 兩者的本質差別
| 顯示值 | 資料意義 | 背後邏輯 |
|---|---|---|
| **空白（Blank）** | 該 Store × Product 組合在 `fact_inventory_snapshot` 的 ENDING 快照中**根本沒有記錄**（這個 SKU 從未在該門店建立庫存檔） | `CALCULATE(SUM(quantity_on_hand), snapshot_type = "ENDING")` 在沒有符合列時回傳 BLANK()  |
| **0.00** | 該組合**有**期末快照記錄，但 `quantity_on_hand = 0`（商品曾經進貨，但期末時真的賣光或歸零） | SUM 有資料列可加總，結果剛好為 0 |
### 業務上代表的含意
- **空白 = Not Carried（未鋪貨）**：這家門店從來沒有經營過這個 SKU，沒有「缺貨」概念，不該算進 Stockout Rate 的分母。
- **0.00 = Out of Stock（真實缺貨）**：這家門店有在賣這個 SKU，但期末庫存為零，這就是你 `Stockout Rate` measure 中 `quantity_on_hand <= 0` 要抓的風險訊號 。
- 你截圖中 store 36、43、45、52 顯示空白，代表 brand 8484 根本沒鋪到這些店；而 store 46 顯示 0.00（且紅字），才是真正的缺貨警示。
### 為什麼 Dead Stock Flag 全部都是 Active
你的 `Dead Stock Flag` 定義為 `IF([Dead Stock Value (90 Days)] > 0, "Dead Stock", "Active")` 。空白與 0 都讓 Dead Stock Value 算不出大於 0 的金額（空白沒庫存、0 沒庫存價值），所以兩者都落入 "Active" 分支，但它們的風險性質完全不同——這正是目前旗標的盲點。
### 改進
在 Page 2 detail table 新增一個更精細的狀態欄位，明確區分三種狀態：

```dax
Inventory Status =
VAR _stock = [Current Stock Qty]
VAR _sales90 = [Sales Qty Last 90 Days]
RETURN
    SWITCH(
        TRUE(),
        ISBLANK(_stock),                          "⚪ Not Carried",
        _stock = 0 && _sales90 > 0,               "🔴 Out of Stock (Active SKU)",
        _stock = 0 && (_sales90 = 0 || ISBLANK(_sales90)), "⚫ Discontinued",
        _stock > 0 && (_sales90 = 0 || ISBLANK(_sales90)), "🟠 Dead Stock",
        "🟢 Healthy"
    )
```

並在 Stockout Rate 的分母中**排除 Not Carried 的組合**，否則會被大量「未鋪貨」稀釋真實缺貨率：

```dax
Stockout Rate (Carried SKU only) =
VAR _oos =
    CALCULATE(
        COUNTROWS('fact_inventory_snapshot'),
        'fact_inventory_snapshot'[quantity_on_hand] <= 0,
        'fact_inventory_snapshot'[snapshot_type] = "ENDING"
    )
VAR _total =
    CALCULATE(
        COUNTROWS('fact_inventory_snapshot'),
        'fact_inventory_snapshot'[snapshot_type] = "ENDING"
    )
RETURN DIVIDE(_oos, _total, BLANK())
```


text
Dead Stock Flag =
IF([Dead Stock Value (90 Days)] > 0, "Dead Stock", "Active")
當 row-level 的 Dead Stock Value = BLANK 時，BLANK() > 0 = FALSE，所以顯示 "Active"——但這不代表該 SKU 真的 Active，只代表這個 brand-store 組合在這個表格沒有庫存快照。這兩種情況（沒庫存 vs 有庫存但沒在動）被混在一起顯示為 "Active"，是語義上不夠精確的地方。

建議的小改進（可選）
text
Dead Stock Flag =
VAR _stock = [Current Stock Qty]
VAR _sales = [Sales Qty Last 90 Days]
VAR _deadVal = [Dead Stock Value (90 Days)]
RETURN
    SWITCH(
        TRUE(),
        ISBLANK(_stock),          BLANK(),           -- 無庫存快照，不判斷
        _deadVal > 0,             "Dead Stock",
        "Active"
    )
這樣沒有庫存紀錄的 row 會顯示 BLANK，而不是誤標 "Active"，表格語義更乾淨。

結論：空白是正常的稀疏資料表現。 


## 核心解釋：BLANK ≠ 0，兩者代表**完全不同的資料狀態**
這個現象**完全正常**，且反映了資料集的真實業務情境。關鍵在於理解 `Current Stock Qty` 的定義：

```dax
Current Stock Qty =
CALCULATE(
    SUM('fact_inventory_snapshot'[quantity_on_hand]),
    'fact_inventory_snapshot'[snapshot_type] = "ENDING"
)
```

這個 Measure 只計算 **`fact_inventory_snapshot` 裡實際存在的 ENDING 快照記錄**。

***
## 三種可能狀態的對照
| 狀態 | fact_inventory_snapshot 是否有紀錄 | quantity_on_hand 值 | Power BI 顯示 |
|---|---|---|---|
| **有庫存** | ✅ 有紀錄 | 5（例如） | `5.00` |
| **賣光但帳上還在** | ✅ 有紀錄 | 0 | `0.00` |
| **根本沒進過貨 / 期末盤點時不存在** | ❌ **無紀錄** | — | **BLANK（空白）** |

你截圖中的 row（例如 store 40, brand 126，Sales 90D = 372）屬於**第三種**：這個 brand-store 組合在 `fact_inventory_snapshot` 的 **2016-12-31 ENDING 快照裡根本沒有一筆對應記錄**。

***
## 為什麼會這樣？符合真實業務邏輯的 4 種情境
### 情境 1：年末前已全部售罄 + 從系統下架
2016 年 Q4 有賣出 372 件 brand 126，但到 12/31 盤點那一刻，這個 SKU 在 store 40 **已經不在系統的庫存表裡**（可能被標為停售、退倉、或 SKU 編號已汰換）。ERP 盤點時不會為「不存在的 SKU」產生一筆 qty=0 的紀錄，而是**整筆 row 不存在**。
### 情境 2：年中才開始銷售、年末才清倉（Transit/Cross-dock 模式）
商品經過 store 40 但**從未在期末快照的那一天停留**。例如：貨物 10 月到店、12 月初賣完，12/31 快照時該 SKU 在這間店的庫存檔已被關閉。
### 情境 3：資料品質問題（ELT 層）
`raw_end_inventory` 只有約 20 萬筆，而銷售紀錄涵蓋的 brand × store 組合遠多於此。**`fact_sales` 的稀疏維度 ≠ `fact_inventory_snapshot` 的稀疏維度**，兩者不是笛卡兒積的完整覆蓋。
### 情境 4：Store 層級差異
brand 126 可能只在**某幾間 store** 有庫存快照，其他 store 雖然有銷售，但期末盤點時庫存為零且系統未保留 qty=0 的 row。

***
## 驗證方法（SQL 直接查）
```sql
-- 驗證 store 40 × brand 126 是否真的沒有 ENDING 快照
SELECT snapshot_type, COUNT(*), SUM(quantity_on_hand)
FROM fact_inventory_snapshot fis
JOIN dim_store s ON fis.store_sk = s.store_sk
JOIN dim_product p ON fis.product_sk = p.product_sk
WHERE s.store_number = 40 AND p.brand = '126'
GROUP BY snapshot_type;
```

若結果**沒有任何 ENDING row**，就證實是「無快照紀錄」而非「qty=0」。

***
## 為什麼 0.00 會顯示為 0.00 而不是 BLANK？
對比前一張截圖（brand 58 store 46，`0.00` 用紅字顯示）：那個 row **有 ENDING 快照紀錄，只是數量是 0**（可能是 ELT 階段 `COALESCE(qty, 0)` 處理過，或原始資料就是 0）。所以 SUM 回傳 0，顯示 0.00。

而目前這張截圖的 BLANK row：**連一筆 ENDING snapshot 紀錄都不存在**，SUM 於空集合回傳 BLANK。

**簡單記憶口訣**：
- `0.00` = 有盤點，但帳上為零
- `BLANK` = 沒有盤點紀錄（這個組合在期末快照表中不存在）

***
## 業務含義：這反而是有價值的洞察
這些 **"有銷售但無期末庫存紀錄"** 的 SKU 其實非常值得留意：

1. **可能是熱銷品已清倉** → 應考慮是否補貨 / 停售決策
2. **可能是資料斷鏈** → ELT 需檢查 `fact_inventory_snapshot` 的覆蓋完整性
3. **Reorder Point 計算風險** → `Current Stock Qty` 為 BLANK 時，`Reorder Alert` Measure 會走 `ISBLANK(_rop) || ISBLANK(_currentStock), BLANK()` 分支，**不會觸發補貨警示**——但實際業務可能需要警示，這是一個潛在盲點。

***
## 建議：在 Page 2 加一個輔助 Measure 偵測這種情況
```dax
Stock Record Status =
VAR _hasSnapshot =
    CALCULATE(
        COUNTROWS('fact_inventory_snapshot'),
        'fact_inventory_snapshot'[snapshot_type] = "ENDING"
    )
VAR _hasSales = [Sales Qty Last 90 Days]
RETURN
    SWITCH(
        TRUE(),
        ISBLANK(_hasSnapshot) && NOT ISBLANK(_hasSales), "⚠️ No Snapshot, Has Sales",
        ISBLANK(_hasSnapshot),                            "No Record",
        _hasSnapshot > 0,                                 "Has Snapshot"
    )
```

把這個欄位加進 Detail Table，就能把你現在問的這類異常 row **顯式標記出來**，發現資料覆蓋缺口並提供監控機制。 


## 直接回答

**`fact_inventory_snapshot` 不是只顯示期末**——它同時包含 BEGINNING 和 ENDING 兩種快照，由 `snapshot_type` 欄位區分。但**你的 `Current Stock Qty` Measure 只讀 ENDING**，所以期初有貨、期末無貨的 SKU 就會在表格中顯示 BLANK。

***

## 資料結構回顧

根據你的 `powerbi_design.ipynb` 設計：

| 來源 raw table | ETL 後進入 | snapshot_type | 時間點 | 筆數 |
|---|---|---|---|---|
| `raw_beg_inventory` | `fact_inventory_snapshot` | `"BEGINNING"` | 2016-01-01 | ~20 萬 |
| `raw_end_inventory` | `fact_inventory_snapshot` | `"ENDING"` | 2016-12-31 | ~20 萬 |

所以 `fact_inventory_snapshot` 總筆數約 **40 萬**，是兩張 raw 表 UNION 後加上 `snapshot_type` 標籤的結果。

***


## 實務意義：四種 SKU 生命週期狀態

在你的資料集中，每個 brand-store 組合會落入以下四種之一：

| BEGINNING | ENDING | 業務解釋 |
|---|---|---|
| ✅ 有 | ✅ 有 | **穩定商品**：全年都在銷售體系內 |
| ✅ 有 | ❌ 無 | **汰換/售罄下架**：年初有、年底退出 |
| ❌ 無 | ✅ 有 | **新品導入**：年中引進的新 SKU |
| ❌ 無 | ❌ 無 | **DSD / Consignment**（如上一題 brand 126 store 40）：完全不走庫存系統 |

***

## 對 Page 2 表格的影響

你現在看到的 BLANK row，其實混合了兩種情況：
1. **期初有、期末無** → 商品已售罄下架
2. **期初期末都無** → 從未進入盤點系統（DSD）

兩者業務意義完全不同，但現行 Measure 都顯示為 BLANK。如果想精準區分，可以加一個輔助 Measure：

```dax
SKU Lifecycle Status =
VAR _hasBeg =
    CALCULATE(
        COUNTROWS('fact_inventory_snapshot'),
        'fact_inventory_snapshot'[snapshot_type] = "BEGINNING"
    )
VAR _hasEnd =
    CALCULATE(
        COUNTROWS('fact_inventory_snapshot'),
        'fact_inventory_snapshot'[snapshot_type] = "ENDING"
    )
RETURN
    SWITCH(
        TRUE(),
        _hasBeg > 0 && _hasEnd > 0, "Stable",
        _hasBeg > 0 && ISBLANK(_hasEnd), "Sold Out / Delisted",
        ISBLANK(_hasBeg) && _hasEnd > 0, "New Introduction",
        "No Inventory Record (DSD?)"
    )
```

把這欄加進 Page 2 Detail Table，一眼就能看出每個 BLANK row 的真實成因，分析深度會明顯提升。

***

## 四種 SKU 生命週期狀態詳解


### 1. `"Stable"` — 穩定商品（兩端都有）

**條件**：`_hasBeg > 0 && _hasEnd > 0`

**意義**：這個 SKU 在 2016 年**年初就存在、年底也還存在**於該門市的庫存系統中。代表這是一個**全年穩定銷售的長青商品**。

**典型例子**：
- 核心暢銷品（Absolut 伏特加、Jack Daniel's 威士忌等主力酒款）
- 常備基礎款（可樂、瓶裝水等日配品）

**業務含義**：
- 這類 SKU 才是 **Inventory Turnover、DSI、Dead Stock** 等 KPI 最適用的對象，因為期初期末都有數據可算「平均庫存」。
- 是作品集中主要分析的目標族群，佔整體 SKU 最多數。

**DAX 判斷**：`_hasBeg > 0` 且 `_hasEnd > 0` 都為 TRUE。

***

### 2. `"Sold Out / Delisted"` — 售罄或下架（只有期初）

**條件**：`_hasBeg > 0 && ISBLANK(_hasEnd)`

**意義**：這個 SKU **年初有庫存，但到年底已完全從庫存系統中消失**。期末盤點時系統找不到這個 brand-store 組合的 row。

**兩種可能子情境**：

**(a) 已售罄且未補貨**
- 期初進了 50 瓶某款季節限定酒，整年賣光後沒再進貨
- 系統把已清零的 SKU 從 store 主檔中刪除（而非保留 qty=0 的 row）

**(b) 主動下架（Delisting）**
- 商品因業績不佳被總部汰換
- 廠商終止合作、品牌退出市場
- 該 store 業態調整（例如超市改成便利店，不再賣大瓶裝酒）

**業務含義**：
- 這類 SKU 是**產品組合優化**分析的重要素材——「哪些商品在年中被清掉了？為什麼？」
- 若 `Sales Qty` 在年後期趨近 0，代表賣光後下架；若突然斷尾，可能是主動下架。
- **不屬於 Dead Stock**（因為期末已經沒庫存），但可分析「**售罄速度**」或「**下架決策合理性**」。

**DAX 判斷**：期初有紀錄，期末紀錄為 BLANK（`COUNTROWS` 回傳 BLANK 不是 0）。

***

### 3. `"New Introduction"` — 新品導入（只有期末）

**條件**：`ISBLANK(_hasBeg) && _hasEnd > 0`

**意義**：這個 SKU **年初還不存在於該門市的庫存系統，但年底已經有紀錄**。代表是 2016 年間**新引進**的商品。

**典型例子**：
- 年中上市的新產品（新品牌、新口味、新包裝規格）
- Store 擴充品類時新增的 SKU
- 廠商新簽約後加入的品牌

**業務含義**：
- 評估**新品導入績效**：這個 SKU 從導入到年底累積賣多少？週轉率如何？
- 注意：此類 SKU 的 **Inventory Turnover 會被高估或低估**——因為 `Average Inventory = (Beg + End) / 2`，但期初 Beg = 0（或 BLANK），平均庫存偏低導致週轉率看起來虛高。分析時應**排除或另外處理**這類 SKU。
- 作品集中可以把這類 SKU 另開一個 "New SKU Performance" 小節展示新品上市半年/季的成效。

**DAX 判斷**：期初 BLANK，期末有紀錄。

***

### 4. `"No Inventory Record (DSD?)"` — 完全無庫存紀錄

**條件**：其他所有情況（即期初期末都 BLANK）

**意義**：這個 brand-store 組合在整張 `fact_inventory_snapshot` 裡**完全找不到任何 row**，無論 BEGINNING 還是 ENDING。但 `fact_sales` 可能有銷售紀錄（例如上一題的 store 40 × brand 126，銷售 372 件但零快照）。

**可能成因**：

**(a) Direct Store Delivery (DSD) 模式**
- 供應商直接送貨到店（Coca-Cola、Pepsi、Frito-Lay、各類酒商 rep）
- 店家只掃 POS 記錄銷售，**庫存由 vendor rep 自己管**，不進總部盤點系統
- 酒類零售（本資料集）在美國很常見 DSD

**(b) Vendor Managed Inventory (VMI) / Consignment 寄售**
- 商品所有權仍歸供應商，賣出才結算
- 門市不把寄售品列入自有庫存盤點

**(c) 資料抽取範圍不對齊**
- `raw_beg_inventory` / `raw_end_inventory` 可能只抽「自營庫存」類別
- `raw_sales` 抽全部 POS 交易（含 DSD 商品）
- 兩邊 SKU-Store 覆蓋範圍天然不同

**(d) Phantom Inventory / 資料品質問題**
- SKU 在 POS 被掃到但 ERP 的庫存主檔從未建檔
- 編號錯置、系統遷移遺漏

**業務含義**：
- 這類 SKU **所有庫存類 KPI（Turnover、DSI、Dead Stock、Reorder Point）都不適用**，因為分母基礎不存在。
- 但「**銷售洞察**」仍然有效（Revenue、Sales Qty、趨勢）。
- 作品集中應**明確標示並說明**這是資料覆蓋的限制，而非 ETL bug——這種誠實的 caveat 是 BI Analyst 成熟度的體現。

**DAX 判斷**：SWITCH 的最後 fallback，捕捉所有不符合前三種條件的情況。技術上等價於 `ISBLANK(_hasBeg) && ISBLANK(_hasEnd)`。

***

## 四種狀態的對照速查

| 狀態 | 期初 | 期末 | 典型佔比 | Turnover 可算? | Dead Stock 可算? | 新品分析 | 風險監控 |
|---|---|---|---|---|---|---|---|
| **Stable** | ✅ | ✅ | 最多（通常 60–80%） | ✅ 最準 | ✅ | ❌ | ✅ 主要對象 |
| **Sold Out / Delisted** | ✅ | ❌ | 中等（10–20%） | ⚠️ 平均庫存失真 | ❌（期末無） | ❌ | ⚠️ 檢查下架合理性 |
| **New Introduction** | ❌ | ✅ | 中小（5–15%） | ⚠️ 被高估 | ✅ | ✅ 主要對象 | ✅ 新品績效追蹤 |
| **No Record (DSD?)** | ❌ | ❌ | 小比例（通常 <10%） | ❌ 無分母 | ❌ | ❌ | ⚠️ 資料覆蓋 caveat |

***

## 為什麼這個分類重要

在沒有 `SKU Lifecycle Status` 之前， Page 2 Table 看到 BLANK 時**無法區分**到底是：
- 已售罄下架（商品層面的結束）
- 從未進入系統（資料層面的缺失）
- 新品但 90 天無銷售（Dead Stock 真實警訊）

加了這個 Measure 後，每一 row 的 BLANK 都有了**具體的業務標籤**，讓表格從「看起來有空白很可疑」進階到「每個空白都有明確的可解釋成因」。

***

### 修復 Scatter Chart「Stockout vs Dead Stock」


#### 加輔助線
- **Analytics pane** → **X-axis constant line** → 設成 Stockout Rate 的平均值（例如 5%）
- **Y-axis constant line** → 設成 Dead Stock % 的平均值（例如 3%）
- 這樣 scatter 自然切成 4 個象限：
  - 右上（高缺貨 + 高呆滯）= **管理失控 🔴**
  - 右下（高缺貨 + 低呆滯）= **補貨問題**
  - 左上（低缺貨 + 高呆滯）= **選品問題**
  - 左下 = **健康 🟢**

#### 修改 Title 與 Subtitle
- Title: `Store Risk Quadrant: Stockout vs Dead Stock`
- Subtitle (動態): 建 Measure
```dax
Title - Risk Scatter =
VAR _n = CALCULATE(DISTINCTCOUNT('dim_store'[store_number]), ALLSELECTED())
RETURN
    "Store Risk Quadrant · " & _n & " Stores · FY 2016"
```

***
